Dylan Ross

Prepare CSV inputs for DIABLO analysis.

## Setup

### Imports

In [29]:
import os
import pickle

import polars as pl

### Constants

In [4]:
# Jupyter server should be running from src/python
CACHE_DIR = os.path.join(
    os.path.dirname(os.path.dirname(os.getcwd())),
    "analysis",
    "dylan",
    "_cache",
)

## Prepare Datasets

### Utility function for preparing the omics block CSV

In [30]:
def prep_subset_for_diablo(
    subset_id: str 
) :
    # load subset sample columns
    with open(os.path.join(CACHE_DIR, f"{subset_id}_cols.pkl"), "rb") as pf:
        wt_cols, mut_cols = pickle.load(pf)

    # load subset omics data
    omics_subset = pl.read_ipc(os.path.join(CACHE_DIR, f"{subset_id}.arrow"))

    # write each block to separate CSVs
    for block in omics_subset["Block"].unique():
        (
            omics_subset
            .filter(pl.col("Block") == block)
            .drop("Block")
            # filter out any rows that have within-group variance below 1e-2
            # this makes DIABLO mad
            .filter(
                (pl.concat_list(wt_cols).list.var() >= 0.01)
                & (pl.concat_list(mut_cols).list.var() >= 0.01)
            )
            .write_csv(os.path.join(
                CACHE_DIR,
                "diablo",
                f"{subset_id}_{block}.csv"
            ))
        )

    # write the target classification label to a CSV
    (
        pl.concat([
            (
                pl.DataFrame({"Sample": wt_cols})
                .with_columns(pl.lit("WT").alias("Target"))
            ), 
            (
                pl.DataFrame({"Sample": mut_cols})
                .with_columns(pl.lit("Mutant").alias("Target"))
            )
        ])
        .write_csv(os.path.join(
            CACHE_DIR,
            "diablo",
            f"{subset_id}_Targets.csv"
        ))
    )

### NPM1 vs. WT
#### Black patients

In [31]:
prep_subset_for_diablo("B_NPM1-WT")

#### White patients

In [32]:
prep_subset_for_diablo("W_NPM1-WT")

### NRAS vs. WT

#### Black patients

In [33]:
prep_subset_for_diablo("B_NRAS-WT")

#### White patients


In [34]:
prep_subset_for_diablo("W_NRAS-WT")